In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    %cd /content/drive/MyDrive/NUIN/CBEM_pytorch
    
import torch
import numpy as np
import matplotlib.pyplot as plt
from utils._raised_cosine_basis import makeRaisedCosBasis
import pandas as pd
# from utils.load_matlab import load_mat_v73, flatten_cell
import utils
from utils import *
# from utils.CBEM import CBEM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: False


In [21]:
data = pd.read_hdf("Data/TemporalNoise1f_v2.h5", key='data')
frame_rate = 60
fd = 1
bin_size = 1/frame_rate
nsims = 20
print("Available cell types: ", data['cell_type'].unique())
cell_type ='OFF transient alpha'

results = []

ctype_df = data.query(f"cell_type == '{cell_type}'").copy().reset_index(drop=True)
print(f"Number of cells of celltype {cell_type}: ", ctype_df['cell_name'].nunique())

for cell_name in ctype_df['cell_name'].unique():
    cell_df = ctype_df.query(f"cell_name == '{cell_name}'").reset_index(drop=True)
    quadrant = cell_df['quadrant'][0]
    spot_size = cell_df['aperture'][0]
    for beta in cell_df['beta'].unique():
        data = cell_df.query(f"beta == {beta}").reset_index(drop=True)
        # -------- training & test sets (once per cell/β) --------
        stim_nr   = np.stack(data.loc[data.noise_seed != 1, 'stimulus'])
        stim_rs   = np.stack(data.loc[data.noise_seed == 1, 'stimulus'])
        spk_nr    = np.stack(data.loc[data.noise_seed != 1, 'spike_train'])
        spk_rs    = np.stack(data.loc[data.noise_seed == 1, 'spike_train'])


Available cell types:  <StringArray>
['OFF transient alpha', 'ON alpha', 'ON delayed', 'OFF sustained alpha']
Length: 4, dtype: str
Number of cells of celltype OFF transient alpha:  4


In [19]:
ctype_df

,animal_id,cell_type,cell_name,retina_id,side,beta,noise_seed,aperture,contrast,mean_level,...,tail_time,total_num_epochs,recorded_side,spike_count,sample_rate,frame_rate,quadrant,stimulus,spike_times,spike_train
